<a href="https://colab.research.google.com/github/XxGhoulPr0xX/SentimentAPI-Equipo31/blob/main/Sentiment_API_Proyecto_ENG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***PROYECTO GRUPO 31***




#🛠️ **Preparación de los Datos**
###📦**Importando Bibilotecas:**
---

**Objetivo:** Definir el dataset y preparar la información para el análisis de sentimientos.  
- Sector de negocio: Atención al cliente / Marketing / Operaciones.  
- Fuente: Reseñas de productos (Amazon Product Reviews).  
- Columnas relevantes: `review_headline`, `review_body`, `sentiment`.  
- Formato esperado: `text` (comentario completo) y `sentiment` (0 = Negativo, 1 = Positivo).  


In [1]:
# 1. TRATAMIENTO DE DATOS Y CONFIGURACIÓN
import pandas as pd
import numpy as np
import re
import joblib
import warnings

# Configuración de alertas
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from pandas.errors import SettingWithCopyWarning
warnings.filterwarnings("ignore", category=SettingWithCopyWarning)

# 2. MODELADO Y PREPROCESAMIENTO (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, roc_auc_score
)

# 3. VISUALIZACIÓN
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
!pip install skl2onnx

from sklearn.pipeline import Pipeline
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 65.0 MB/s eta 0:00:00


###📌 **Extracción**
---

**Acciones a realizar:**  
- Carga del dataset con `pandas.read_csv()`.  
- Revisión inicial con `df.info()` y `df.head()`.  
- Confirmación de columnas y tamaño del dataset.  


In [3]:
#cargar el archivo de datos
dts = pd.read_csv('https://raw.githubusercontent.com/XxGhoulPr0xX/SentimentAPI-Equipo31/refs/heads/main/dataset/Amazon-Product-Reviews%20-%20Amazon%20Product%20Review.csv')
dts.head()


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,sentiment
0,US,11555559,R1QXC7AHHJBQ3O,B00IKPX4GY,2693241,"Fire HD 7, 7"" HD Display, Wi-Fi, 8 GB",PC,5,0,0,N,Y,Five Stars,Great love it,2015-08-31,1
1,US,31469372,R175VSRV6ZETOP,B00IKPYKWG,2693241,"Fire HD 7, 7"" HD Display, Wi-Fi, 8 GB",PC,3,0,0,N,N,Lots of ads Slow processing speed Occasionally...,Lots of ads<br />Slow processing speed<br />Oc...,2015-08-31,0
2,US,26843895,R2HRFF78MWGY19,B00IKPW0UA,2693241,"Fire HD 7, 7"" HD Display, Wi-Fi, 8 GB",PC,5,0,0,N,Y,Well thought out device,Excellent unit. The versatility of this table...,2015-08-31,1
3,US,19844868,R8Q39WPKYVSTX,B00LCHSHMS,2693241,"Fire HD 7, 7"" HD Display, Wi-Fi, 8 GB",PC,4,0,0,N,N,Not all apps/games we were looking forward to ...,I bought this on Amazon Prime so I ended up bu...,2015-08-31,1
4,US,1189852,R3RL4C8YP2ZCJL,B00IKPZ5V6,2693241,"Fire HD 7, 7"" HD Display, Wi-Fi, 8 GB",PC,5,0,0,N,Y,Five Stars,All Amazon products continue to meet my expect...,2015-08-31,1


In [ ]:
dts.info()

In [ ]:
dts.dtypes

# 📝 **Limpieaza de los Datos**
###❌ **Eliminamos las columnas innecesarias:**
---

In [ ]:
#En este caso lo haremos de manera inversa (seleccionar las columnas necesarias) y guardamos el resultado obtenido en una nueva variable:
dts2 = dts[["review_body", "sentiment"]].copy()
dts2

In [ ]:
#Confirmamos columnas y tamaño de ambos:
print(dts.columns)
print(dts.shape)
print(dts2.columns)
print(dts2.shape)


### 🧩 **Unión de las columnas (review_headline) y (review_body)**
---


**Acciones a realizar:**  
- Se creó la columna `text` concatenando `review_body`.  
- Se simplificó la estructura del DataFrame conservando únicamente `text` y `sentiment`.  
- Validación de longitud y ejemplos aleatorios para asegurar coherencia.  


In [ ]:
#Se crea una nueva columna llamada text concatenando el título (review_headline) y el cuerpo de la reseña (review_body).
#Luego, se conserva únicamente esta columna de texto junto con la etiqueta sentiment, simplificando la estructura del DataFrame.
#De esta manera el modelo leerá un solo texto completo, como lo haría una persona.
dts2["text"] = dts2["review_body"]
dts2 = dts2[["text", "sentiment"]]
dts2

###🔍 **Detección de textos vacíos**
---

**Acciones a realizar:**  
- Identificación de registros sin contenido en la columna `text`.  
- Se encontraron y marcaron filas con valores nulos o longitud cero.  


In [ ]:
#Verificaremos si existen datos vacíos en la nueva columna creada:
dts2[dts2["text"].isna()]

 ### 🧹 **Eliminación de textos vacíos**
 ---

**Acciones a realizar:**  
- Se eliminaron los registros vacíos para evitar errores futuros en el modelo.  
- Se verificó nuevamente que no existieran valores nulos en `text`.  


In [ ]:
#Con los 6 registros encontrados como vacíos, vamos a eliminarlos para no obtener errores en el futuro con el modelo a entrenar.
dts2 = dts2[dts2["text"].str.len() > 0]
dts2

In [ ]:
#Volvemos a verificar que no existan más datos vacíos
dts2[dts2["text"].isna()]


In [ ]:
print(dts2.shape)

### 🧼 **Limpieza del texto**
---

**Acciones a realizar:**  
- Conversión de texto a minúsculas.  
- Eliminación de URLs, caracteres especiales y espacios innecesarios.  
- Resultado: texto normalizado y listo para vectorización.  


In [ ]:
#Convierte el texto a minúsculas y elimina enlaces, caracteres especiales y espacios innecesarios, dejando únicamente
#palabras útiles para el análisis de sentimiento.
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

dts2["text"] = dts2["text"].apply(clean_text)
dts2


In [ ]:
conteo = dts2['sentiment'].value_counts()
conteo

Clases con desbalanceo, hay que hacer un metodo para que ambas clases esten balanceadas, se usará Undersampling

In [ ]:
# 1. Separar las clases
df_clase_1 = dts2[dts2['sentiment'] == 1]
df_clase_0 = dts2[dts2['sentiment'] == 0]

# 2. Submuestrear la clase mayoritaria (1)
# El número de muestras será igual al tamaño de la clase 0 (5078)
df_clase_1_downsampled = df_clase_1.sample(n=len(df_clase_0), random_state=42)

# 3. Concatenar de nuevo
df_balanceado = pd.concat([df_clase_1_downsampled, df_clase_0])

# Mezclar las filas para que no queden ordenadas por clase
df_balanceado = df_balanceado.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanceado['sentiment'].value_counts())

#🧮 **Transformación de texto a números**
### 🔢 **TF-IDF**
---


**Acciones a realizar:**  
- Implementación de `TfidfVectorizer`.  
- Configuración de parámetros: `max_features=5000`, `ngram_range=(1,2)`, `stop_words='english'`.  
- Transformación de los textos a vectores numéricos.  


In [ ]:
#“TF-IDF transforma el texto en vectores numéricos para que después el modelo supervisado aprenda a asociar patrones de palabras con
#etiquetas de sentimiento.”
# Inicializamos el vectorizador TF-IDF

tfidf = TfidfVectorizer(
    max_features=5000, #número máximo de palabras a considerar
    ngram_range=(1,2),
    stop_words="english" #elimina palabras vacías
)

# Ajustamos y tranformamos los textos
X = tfidf.fit_transform(df_balanceado["text"])   #df['text'] es la columna combinada
y = df_balanceado["sentiment"]                   #Etiquetas de sentimiento del nuevo DF

# Revisión de la forma de la matriz
print("Shape de la matriz TF-IDF:", X.shape)

# 🤖 **Modelo — Entrenamiento supervisado**
### ✂️ **Separación de datos (Train / Test Split)**
---

**Acciones a realizar:**  
- División de datos en entrenamiento y prueba (`train_test_split`).  
- Inicialización de `LogisticRegression(max_iter=1000, n_jobs=-1)`.  
- Entrenamiento del modelo supervisado con los textos vectorizados.  


In [ ]:
#Divide los datos en entrenamiento y prueba para evaluar el modelo correctamente.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 🧠 **Inicialización y entrenamiento del modelo (Logistic Regression)**
---



In [ ]:
#Crea y entrena el modelo supervisado que aprende el sentimiento.
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
)

model.fit(X_train, y_train)

#**Guardado**
---

**Acciones a realizar:**  
- Serialización del pipeline con `joblib.dump()`.  
- Archivo generado: `modelo_sentimiento.pkl`.  
- Prueba de carga exitosa con `joblib.load()`.  


In [ ]:
joblib.dump(model, 'modelo_sentimientoENG.pkl')

print("Modelo guardado con éxito.")

In [ ]:
# 1. Crear un pipeline que combine ambos procesos
pipeline_sentimiento = Pipeline([
    ('tfidf', tfidf),
    ('clf', model)
])

# 2. Entrenar el pipeline completo
# (Nota: tfidf ya debe estar ajustado o puedes usar fit con los datos crudos)
pipeline_sentimiento.fit(df_balanceado["text"], df_balanceado["sentiment"])

# 3. Definir la entrada como String (porque el pipeline ahora recibe texto)
# [None, 1] significa que puede recibir N filas de 1 columna de texto
initial_type = [('string_input', StringTensorType([None, 1]))]

# 4. Convertir a ONNX (con opciones específicas para TF-IDF)
options = {id(pipeline_sentimiento): {'zipmap': False}} # Opcional: evita una estructura compleja de salida
onx = convert_sklearn(pipeline_sentimiento, initial_types=initial_type, options=options)

# 5. Guardar
with open("modelo_sentimientoENG.onnx", "wb") as f:
    f.write(onx.SerializeToString())

#📈 **Métricas**
### 📊 **Métricas de desempeño**
---

**Acciones a realizar:**  
- Evaluación en conjunto de prueba.  
- Métricas calculadas: Accuracy, Precision, Recall, F1-Score.  
- Reporte de clasificación generado con `classification_report`.  


In [ ]:
# Celda: Cargar el modelo
model_path = 'modelo_sentimientoENG.pkl'
modelo = joblib.load(model_path)

print("¡Modelo cargado exitosamente con joblib!")

In [ ]:
# Celda 2: Inspeccionar el contenido
if isinstance(modelo, dict):
    print("El archivo contiene un diccionario con claves:")
    print(list(modelo.keys()))
    # ver más detalles de cada clave
    for key, value in modelo.items():
        print(f"\nClave: {key}")
        print(f"Tipo: {type(value)}")
        if hasattr(value, 'shape'):
            print(f"Shape: {value.shape}")
        elif isinstance(value, str):
            print(f"Valor: {value[:100]}...")  # primeros 100 caracteres si es texto

elif hasattr(modelo, 'steps'):  # Es un Pipeline
    print("Es un Pipeline de scikit-learn. Pasos del pipeline:")
    for name, step in modelo.steps:
        print(f"- {name}: {type(step).__name__}")
        # Detalles adicionales por paso
        if hasattr(step, 'get_params'):
            print("   Parámetros:", step.get_params())
        if hasattr(step, 'classes_'):
            print("   Clases detectadas:", step.classes_)
        if hasattr(step, 'n_features_in_'):
            print("   Número de características:", step.n_features_in_)

elif hasattr(modelo, 'predict'):  # Es un clasificador directo
    print("Es un clasificador directo.")
    print("Clases:", getattr(modelo, 'classes_', 'No disponible'))
    print("Parámetros:", modelo.get_params())

else:
    print("Tipo desconocido. Detalles manuales:")
    print(dir(modelo))  # muestra todos los atributos y métodos

In [ ]:
# Usar las variables
y_true = y_test.values if hasattr(y_test, 'values') else y_test
y_pred = modelo.predict(X_test)

# Verificación rápida
print("Clases únicas en y_true:", np.unique(y_true))
print("Clases únicas en y_pred:", np.unique(y_pred))
print(f"Tamaño de y_true: {len(y_true)}, y_pred: {len(y_pred)}")

# 1. Accuracy - Precisión
accuracy = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# 2. Reporte completo (precision, recall, f1-score)
class_names = ['Negativo (0)', 'Positivo (1)']

print("\nReporte de Clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

Modelo de clasificación binaria (clases 0 = Negativo y 1 = Positivo), evaluado en un conjunto de prueba de 2032 ejemplos (balanceado: 1016 negativos y 1016 positivos).

**Accuracy (Exactitud General)**

Valor: 0.8642 (86.42%).

**Explicación:** El modelo acierta en el 86.42% de los casos en total. Es decir, de los 2032 ejemplos, predijo correctamente 1756 (896 negativos + 860 positivos).

**Reporte de Clasificación (Precision, Recall y F1-Score)**
Este reporte mide el desempeño por clase. Para cada métrica:

*  **Precision:** De los ejemplos que el modelo predijo como esa clase, ¿cuántos son realmente correctos? (Evita falsos positivos).

*  **Recall:** De los ejemplos verdaderos de esa clase, ¿cuántos capturó el modelo? (Evita falsos negativos).

* **F1-Score:** Media armónica de precision y recall (balancea ambos; útil si las clases son desiguales, aunque aquí están balanceadas).

* **Support:** Número de ejemplos reales por clase (1016 cada una).

**Explicación por clase:**

* **Negativo (0):** Buen recall (88.19% → detecta bien la mayoría de negativos reales), pero precision un poco más baja (85.17% → ~15% de lo que predice como negativo son falsos positivos). F1 equilibrado en 86.65%.
* **Positivo (1):** Precision alta (87.76% → pocas quejas falsas), pero recall un poco menor (84.65% → pierde ~15% de positivos reales). F1 en 86.17%.

* **Macro Avg:** Promedio simple por clase (ignora el tamaño). Casi igual al accuracy, lo que indica balance bueno.

* **Weighted Avg:** Promedio ponderado por support (aquí igual al macro porque las clases están equilibradas).

**En resumen:** El modelo es equilibrado entre clases, con un leve sesgo a favor de detectar negativos (mejor recall para 0). Ideal para atención al cliente, donde priorizar quejas (negativos) es clave.

In [ ]:
# Variables
y_true = y_test.values if hasattr(y_test, 'values') else y_test
y_pred = y_pred

# Obtener probabilidades para la clase positiva (1)
probas = modelo.predict_proba(X_test)[:, 1]  # Probabilidad de clase 1 (positivo)

# ===================== Métrica 1: Curva ROC y AUC =====================
fpr, tpr, _ = roc_curve(y_true, probas)
roc_auc = auc(fpr, tpr)

# Valor directo de AUC (más simple)
auc_value = roc_auc_score(y_true, probas)
print(f"AUC (Área bajo la curva ROC): {auc_value:.4f} ({auc_value*100:.2f}%)")

# Gráfica
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')  # línea diagonal de azar
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()



**Curva ROC y AUC (Área bajo la curva ROC):** mide la capacidad del modelo para distinguir entre clases positivas y negativas.

**AUC (0.5 = azar, 1.0 = perfecto):**
* 0.9: excelente
* 0.8–0.9: muy bueno
* 0.7–0.8: aceptable

El modelo tiene un buen AUC si está por encima de 0.92–0.93 (típico para buenos modelos de sentimiento).

In [ ]:
# ===================== Métrica 2: Curva Precision-Recall =====================
precision, recall, _ = precision_recall_curve(y_true, probas)
avg_precision = average_precision_score(y_true, probas)

print(f"AP (Average Precision): {avg_precision:.4f} ({avg_precision*100:.2f}%)")

# Gráfica
plt.figure(figsize=(7, 6))
plt.step(recall, precision, where='post', color='b', alpha=0.7,
         label=f'AP = {avg_precision:.4f}')
plt.fill_between(recall, precision, step='post', alpha=0.2, color='b')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Curva Precision-Recall')
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.show()

**Precision-Recall Curve (curva de precisión-recall):** especialmente útil cuando las clases están desbalanceadas o interesa más la detección de la clase positiva.

**Average Precision (AP):**
 - Es el área bajo la curva Precision-Recall.
 - Muy útil si la clase positiva (ej. "negativo" = quejas) es minoritaria o    se quiere priorizar precisión.
 - Valores cercanos a 1 son ideales.

In [ ]:
# Importaciones necesarias
from sklearn.metrics import log_loss, brier_score_loss, matthews_corrcoef, cohen_kappa_score, fbeta_score

print("\nMétricas adicionales:")

# 1. Log Loss (menor es mejor)
ll = log_loss(y_true, probas)
print(f"Log Loss (Binary Cross-Entropy): {ll:.4f}")

# 2. Brier Score (menor es mejor, similar a MSE de las probabilidades)
bs = brier_score_loss(y_true, probas)
print(f"Brier Score: {bs:.4f}")

# 3. Matthews Correlation Coefficient (MCC) - muy equilibrado
mcc = matthews_corrcoef(y_true, y_pred)
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

# 4. Cohen's Kappa (acuerdo corregido por azar)
kappa = cohen_kappa_score(y_true, y_pred)
print(f"Cohen's Kappa: {kappa:.4f}")

# 5. F-beta con β=2 (da más peso al recall)
fbeta = fbeta_score(y_true, y_pred, beta=2)
print(f"F-beta Score (β=2, más peso al recall): {fbeta:.4f}")

**Métricas Adicionales**

1. **Matthews Correlation Coefficient (MCC)** → es una de las métricas más equilibradas y recomendadas en papers recientes para clasificación binaria (mejor que accuracy cuando hay balance).  **Valor ideal: -1 a +1 (>0.5 bueno)**

2. **Log Loss (Binary Cross-Entropy) o Brier Score** → evaluar la calidad de las probabilidades/confianza, estas métricas evalúan qué tan bien calibradas están. **Valor ideal: menor mejor /  < 0.25**

3. **Cohen's Kappa** → Mide acuerdo más allá del azar (útil para comparar con baseline) **Valor ideal:  > 0.6 bueno**

4. **F-beta (β=2)** → si el objetivo de negocio es no perder comentarios negativos (quejas), esta métrica prioriza el recall de la clase negativa. **Valor ideal: > 0.85**


### 🧮 **Matriz de confusión**
---

**Acciones a realizar:**  
- Generación de matriz de confusión con `confusion_matrix`.  
- Visualización con `seaborn.heatmap`.  
- Interpretación:  
  - Verdaderos negativos y positivos correctamente clasificados.  
  - Identificación de falsos positivos y falsos negativos.  


In [ ]:
# 3. Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel('Predicciones')
plt.ylabel('Etiquetas Verdaderas')
plt.title('Matriz de Confusión')
plt.show()

# Opcional: Matriz normalizada (porcentajes por fila)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel('Predicciones')
plt.ylabel('Etiquetas Verdaderas')
plt.title('Matriz de Confusión Normalizada')
plt.show()

Esta grafica resume cuántos ejemplos clasificó cada clase vs. cuántos eran realmente. Se lee como:

* Filas: Etiquetas verdaderas (lo que era real).
* Columnas: Predicciones del modelo (lo que predijo).

**Matriz Absoluta (números crudos):**

**Totales:**

* Predijo 1016 negativos (896 + 156) y 1016 positivos (120 + 860).
* Aciertos: 896 + 860 = 1756 (86.42%).
* Errores: 120 FP (predijo positivo cuando era negativo) + 156 FN (predijo negativo cuando era positivo) = 276 (13.58%).

**Matriz Normalizada (porcentajes por fila, suma 100% horizontalmente):**}

**Explicación:**

- Para negativos reales: El modelo los clasifica correctamente en 88.19% (buena detección de quejas).
- Para positivos reales: Correcto en 84.65% (pierde 15.35% como falsos negativos, quizás por textos ambiguos).
- Errores comunes: Más FP en negativos (11.81% de negativos se "pierden" como positivos) y FN en positivos (15.35%).

# 📊 Conclusiones Finales del Modelo de Análisis de Sentimientos

**Resumen del desempeño:**
- El modelo entrenado con **TF-IDF + Regresión Logística** alcanzó una **exactitud (Accuracy) de ~86%**, lo que indica un buen nivel de predicción en el conjunto de prueba.
- La **precisión y el recall** muestran un equilibrio aceptable entre clases:
  - **Clase Negativa (0):** recall alto (~88%), lo que significa que el modelo detecta correctamente la mayoría de las quejas.
  - **Clase Positiva (1):** precisión alta (~87%), lo que reduce la cantidad de falsos positivos.
- El **F1-Score** se mantiene alrededor de 0.86 en ambas clases, confirmando un desempeño balanceado.

**Interpretación de métricas:**
- El modelo es ligeramente mejor identificando comentarios negativos, lo cual es valioso para el negocio, ya que permite priorizar la atención a clientes insatisfechos.
- Los falsos negativos (comentarios positivos clasificados como negativos) y falsos positivos (comentarios negativos clasificados como positivos) se mantienen en niveles manejables.

**Valor para el negocio:**
- Permite **automatizar la clasificación de reseñas y comentarios**, reduciendo la necesidad de lectura manual.
- Facilita la **detección temprana de quejas**, mejorando la capacidad de respuesta del área de atención al cliente.
- Proporciona métricas cuantitativas para **monitorear campañas de marketing** y evaluar la percepción de la marca a lo largo del tiempo.

**Conclusión general:**
El modelo cumple con los objetivos del MVP: ofrece una solución simple y funcional para clasificar sentimientos en textos de clientes, con métricas sólidas y potencial de integración inmediata en una API de análisis de sentimientos.
